# 🚀 LORCEN-RAG: Production QLoRA Fine-Tuning Pipeline for GATE CS
### Domain-Adapted Mathematical Reasoning Engine on `Qwen2.5-1.5B-Instruct`

This notebook trains a **4-Bit QLoRA (Quantized Low-Rank Adaptation)** reasoning adapter on authentic **GATE Computer Science & IT (1990 - 2026)** problems.

#### ⚙️ Technical Architecture:
- **Base Model**: `Qwen/Qwen2.5-1.5B-Instruct`
- **Quantization**: 4-bit NormalFloat (NF4) with Double Quantization (`bitsandbytes`)
- **PEFT/LoRA**: Rank $r=16$, Alpha $\alpha=32$, Dropout $0.05$
- **Target Modules**: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`
- **Prompt Template**: ChatML `<|im_start|>system...<|im_start|>user...<|im_start|>assistant...`

## 1. Clean Environment & Install Core LLM Dependencies

In [ ]:
# 1. Uninstall mismatched/outdated optional packages in Colab
!pip uninstall -y torchvision torchaudio torchao

# 2. Install core LLM NLP training packages
!pip install -q -U "transformers>=4.48.0" datasets peft bitsandbytes "trl>=0.12.0" accelerate

## 2. Verify GPU Acceleration

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 3. Clone Repository & Extract Real Training Dataset

In [ ]:
%cd /content
!rm -rf LORCEN-RAG
!git clone https://github.com/piyush23-eng/LORCEN-RAG.git
%cd /content/LORCEN-RAG
!python scripts/prepare_training_data.py

## 4. Run QLoRA Fine-Tuning

In [ ]:
%cd /content/LORCEN-RAG
!python scripts/train_qlora.py \
    --base_model Qwen/Qwen2.5-1.5B-Instruct \
    --data_path data/train_gate_cs_dataset.jsonl \
    --output_dir models/calypso_gate_qlora \
    --epochs 4 \
    --batch_size 2 \
    --grad_accum 4 \
    --lr 2e-4

## 5. Direct Native PyTorch Inference with Fine-Tuned Adapter

In [ ]:
import sys
sys.modules['torchvision'] = None
sys.modules['torchaudio'] = None
sys.modules['torchao'] = None

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "/content/LORCEN-RAG/models/calypso_gate_qlora/final_adapter"

print("Loading tokenizer and base model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Attaching fine-tuned QLoRA adapter...")
trained_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
trained_model.eval()

# Formulate Test Query with ChatML template
prompt = (
    "<|im_start|>system\nYou are an expert GATE Computer Science reasoning assistant.<|im_end|>\n"
    "<|im_start|>user\n"
    "Consider a hard disk with a rotational speed of 15000 rpm. The time to move the read/write head from a track to its adjacent track is 1 millisecond. "
    "Initially, the head is on track 0. The number of sectors per track is 400. "
    "Transfer data from 10 randomly located sectors in each of tracks in order: 5, 12 and 7. "
    "What is the total data transfer time in milliseconds?<|im_end|>\n"
    "<|im_start|>assistant\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")

with torch.no_grad():
    output_tokens = trained_model.generate(
        **inputs,
        max_new_tokens=400,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

decoded_output = tokenizer.decode(output_tokens[0], skip_special_tokens=False)
solution = decoded_output.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

print("\n================ 🎯 MODEL GENERATED STEP-BY-STEP SOLUTION ================")
print(solution)
print("===========================================================================")